# Adaptive Patch Size ViT + Structured Magnitude-Based Pruning

Modifikasi Arsitektur Vision Transformer Menggunakan *Adaptive Patch Size* yang Dioptimalkan melalui *Structured Magnitude-Based Pruning* untuk Klasifikasi Patologi Daun pada Kondisi Pencahayaan Dinamis.

Notebook ini memanggil seluruh modul di `adaptive_vit/` (library generik) dan `examples/plantvillage_taxonomy.py` (adapter khusus dataset PlantVillage) secara berurutan -- sama persis dengan urutan Cell 1-10 di `examples/plantvillage_pipeline_example.py`, hanya dipecah jadi cell notebook sungguhan supaya bisa dijalankan langsung di Colab, cell demi cell.

**Sebelum menjalankan**: pastikan folder `adaptive_vit_pkg/` (berisi `adaptive_vit/` dan `examples/`) sudah ada di Google Drive kamu, dan dataset PlantVillage (Updated) sudah diunduh/diekstrak ke Colab (lihat Cell 0b).

## Cell 0a -- Hubungkan Google Drive & tambahkan package ke `sys.path`

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import sys, os

# --- SESUAIKAN path ini dengan lokasi folder adaptive_vit_pkg/ di Drive kamu ---
PACKAGE_ROOT = '/content/drive/MyDrive/skripsi/adaptive_vit_pkg'

sys.path.append(PACKAGE_ROOT)                              # supaya `import adaptive_vit` jalan
sys.path.append(os.path.join(PACKAGE_ROOT, 'examples'))    # supaya `import plantvillage_taxonomy` jalan

print('adaptive_vit ditemukan di:', PACKAGE_ROOT)
print(os.listdir(PACKAGE_ROOT))

## Cell 0b -- Siapkan dataset

Sesuaikan sel ini dengan cara kamu membawa dataset PlantVillage (Updated) ke Colab (unduh via Kaggle API, atau ekstrak dari Drive/zip yang sudah diunggah). `DATASET_ROOT` di bawah harus menunjuk ke folder yang di dalamnya langsung berisi folder per spesies (Apple, Tomato, dst).

In [ ]:
# Contoh kalau dataset sudah berupa .zip di Drive:
# import zipfile
# with zipfile.ZipFile('/content/drive/MyDrive/skripsi/plant-village-dataset-updated.zip') as z:
#     z.extractall('/content/plant-village-dataset-updated')

DATASET_ROOT = '/content/plant-village-dataset-updated'

## Cell 0c -- Import modul & konfigurasi

In [ ]:
import functools

import torch
from torch.utils.data import DataLoader
from PIL import Image

from adaptive_vit import (
    DataConfig, ICSConfig, ModelConfig, TrainConfig, PruningConfig,
    ModifiedVisionTransformer, BaselineVisionTransformer,
    adaptive_collate_fn, train_model,
)
from adaptive_vit import data as avdata
from adaptive_vit import preprocessing, ics, evaluate, pruning, visualize

from plantvillage_taxonomy import scan_plantvillage_dataset

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

data_cfg = DataConfig(dataset_root=DATASET_ROOT)
ics_cfg = ICSConfig()
train_cfg = TrainConfig()
prune_cfg = PruningConfig()

# adaptive_collate_fn's region_size HARUS sama dengan region_size yang dipakai
# dataset untuk membangun patch_size_map-nya (ics_cfg.region_size) -- DataLoader
# cuma manggil collate_fn dengan satu argumen (batch), jadi region_size diikat di sini.
collate_fn = functools.partial(adaptive_collate_fn, region_size=ics_cfg.region_size)

## Cell 1 -- Scan & validasi dataset

`check_near_duplicates=True` mengaktifkan deteksi gambar duplikat termasuk versi yang di-*flip* (bukan cuma duplikat persis byte-nya) -- opsional, agak lebih lambat tapi lebih aman dari kebocoran data train/test. Nonaktifkan (`False`) kalau ingin validasi cepat dulu.

In [ ]:
samples, class_names = scan_plantvillage_dataset(data_cfg.dataset_root)
valid_samples, report = avdata.validate_dataset(
    samples, class_names, min_resolution=data_cfg.image_size,
    imbalance_tolerance=data_cfg.class_imbalance_tolerance,
    check_near_duplicates=False,
)
print('Classes exceeding the imbalance tolerance:', report['classes_exceeding_tolerance'])

model_cfg = ModelConfig(num_classes=len(class_names))  # -> 29 untuk studi ini

## Cell 2 -- Stratified split 80:10:10

In [ ]:
train_samples, val_samples, test_samples = avdata.stratified_split(
    valid_samples, ratios=data_cfg.split_ratios, seed=data_cfg.split_seed,
)

## Cell 3 -- Hitung mean/std aktual dari subset TRAIN

In [ ]:
pre_tf_novaug = preprocessing.build_pre_normalize_transform(
    data_cfg.image_size, split='test', apply_lighting_simulation=False)

class _RawImageDataset(torch.utils.data.Dataset):
    def __init__(self, samples_):
        self.samples = samples_
    def __len__(self):
        return len(self.samples)
    def __getitem__(self, idx):
        img = Image.open(self.samples[idx].filepath).convert('RGB')
        return pre_tf_novaug(img), self.samples[idx].label

mean, std = preprocessing.compute_dataset_mean_std(_RawImageDataset(train_samples))
print('Training-subset mean/std:', mean, std)

## Cell 4 -- Fit alpha, beta, p30, p70 dari sampel subset VALIDASI

In [ ]:
sample_val = val_samples[:400]
val_images_np = [
    pre_tf_novaug(Image.open(s.filepath).convert('RGB')).permute(1, 2, 0).numpy()
    for s in sample_val
]
alpha, beta = ics.fit_alpha_beta(val_images_np, region_size=ics_cfg.region_size)
p30, p70 = ics.fit_percentile_thresholds(
    val_images_np, alpha, beta, region_size=ics_cfg.region_size,
    p_low=ics_cfg.p_low, p_high=ics_cfg.p_high,
)

## Cell 5 -- Bangun Dataset & DataLoader final (train/val/test)

In [ ]:
def make_dataset(samples_, split):
    return avdata.ImageListDataset(
        samples_,
        pre_normalize_transform=preprocessing.build_pre_normalize_transform(
            data_cfg.image_size, split=split, apply_lighting_simulation=(split == 'train')),
        normalize_transform=preprocessing.build_normalize_transform(mean, std),
        ics_alpha=alpha, ics_beta=beta, ics_p30=p30, ics_p70=p70,
        region_size=ics_cfg.region_size,
    )

train_ds = make_dataset(train_samples, 'train')
val_ds = make_dataset(val_samples, 'val')
test_ds = make_dataset(test_samples, 'test')

train_loader = DataLoader(train_ds, batch_size=train_cfg.batch_size, shuffle=True,
                           collate_fn=collate_fn, num_workers=4)
val_loader = DataLoader(val_ds, batch_size=train_cfg.batch_size, shuffle=False,
                         collate_fn=collate_fn, num_workers=2)
test_loader = DataLoader(test_ds, batch_size=train_cfg.batch_size, shuffle=False,
                          collate_fn=collate_fn, num_workers=2)

## Cell 6 -- Latih Modified Vision Transformer (kontribusi utama: Adaptive Patch Size)

In [ ]:
modified_model = ModifiedVisionTransformer(model_cfg)
modified_model, history = train_model(modified_model, train_loader, val_loader, train_cfg, device)

pre_prune_metrics = evaluate.compute_classification_metrics(modified_model, val_loader, device)
print('Modified ViT accuracy (before pruning) on val:', pre_prune_metrics.accuracy)

## Cell 7 -- Latih Baseline ViT (fixed 16x16 patch, untuk pembanding di Tabel 3.2/3.3)

In [ ]:
baseline_model = BaselineVisionTransformer(model_cfg)
baseline_train_loader = DataLoader(avdata.PlainImageDataset(train_ds), batch_size=train_cfg.batch_size,
                                    shuffle=True, num_workers=4)
baseline_val_loader = DataLoader(avdata.PlainImageDataset(val_ds), batch_size=train_cfg.batch_size,
                                  shuffle=False, num_workers=2)
baseline_model, _ = train_model(baseline_model, baseline_train_loader, baseline_val_loader, train_cfg, device)

## Cell 8 -- Structured Magnitude-Based Pruning + fine-tuning (kontribusi kedua: optimasi/efisiensi)

In [ ]:
prune_result = pruning.iterative_prune_and_finetune(
    modified_model,
    baseline_accuracy=pre_prune_metrics.accuracy,
    train_loader=train_loader, val_loader=val_loader,
    cfg=prune_cfg, device=device,
)
final_model = prune_result.model
print(f'Final prune ratio: {prune_result.final_ratio:.2%}, '
      f'val_acc: {prune_result.val_accuracy:.4f}')

## Cell 9 -- Evaluasi di TEST SET (akurasi, efisiensi, ketahanan pencahayaan dinamis)

In [ ]:
lighting_test_ds = avdata.ImageListDataset(
    test_samples,
    pre_normalize_transform=preprocessing.build_lighting_only_transform(data_cfg.image_size),
    normalize_transform=preprocessing.build_normalize_transform(mean, std),
    ics_alpha=alpha, ics_beta=beta, ics_p30=p30, ics_p70=p70,
    region_size=ics_cfg.region_size,
)
lighting_loader = DataLoader(lighting_test_ds, batch_size=train_cfg.batch_size, shuffle=False,
                              collate_fn=collate_fn, num_workers=2)
baseline_lighting_loader = DataLoader(avdata.PlainImageDataset(lighting_test_ds), batch_size=train_cfg.batch_size,
                                       shuffle=False, num_workers=2)
baseline_test_loader = DataLoader(avdata.PlainImageDataset(test_ds), batch_size=train_cfg.batch_size,
                                   shuffle=False, num_workers=2)

comparison_report = evaluate.compare_baseline_vs_optimized(
    baseline_model=baseline_model,
    optimized_model=final_model,
    baseline_test_loader=baseline_test_loader,
    optimized_test_loader=test_loader,
    baseline_lighting_loader=baseline_lighting_loader,
    optimized_lighting_loader=lighting_loader,
    device=device,
)

## Cell 10 -- Visualisasi untuk BAB IV

In [ ]:
visualize.plot_training_curves(history, save_path='fig_training_curves.png')
visualize.plot_pruning_search(prune_result.history, save_path='fig_pruning_search.png')
visualize.plot_model_comparison(comparison_report, save_path='fig_model_comparison.png')

# Peta ukuran patch adaptif untuk satu sampel gambar test, menunjukkan mekanisme
# inti ICS. Sengaja mengambil ULANG gambar pra-normalisasi (pre_tf_novaug), BUKAN
# tensor dari test_ds[0] -- tensor itu sudah lewat normalize_transform (mean/std
# ImageNet) yang bisa bernilai negatif dan akan ke-clip jadi hitam/putih kalau
# langsung divisualisasikan.
sample = test_samples[0]
sample_raw = Image.open(sample.filepath).convert('RGB')
sample_pre_tensor = pre_tf_novaug(sample_raw)
sample_image_np = sample_pre_tensor.permute(1, 2, 0).numpy()
sample_patch_map = ics.compute_patch_size_map_for_image(
    sample_image_np, alpha, beta, p30, p70, region_size=ics_cfg.region_size,
)
visualize.visualize_patch_size_map(sample_image_np, sample_patch_map,
                                    region_size=ics_cfg.region_size,
                                    save_path='fig_patch_size_map.png')